In [ ]:
# NeuroScope — one-cell end-to-end run on Kaggle (GPU T4 x2, Internet On, inputs: brats20 + neuroscope-code)
import os, shutil
SRC = "/kaggle/input/neuroscope-code"
if os.path.isdir(f"{SRC}/neuroscope"): SRC = f"{SRC}/neuroscope"
if not os.path.isdir("/kaggle/working/neuroscope"): shutil.copytree(SRC, "/kaggle/working/neuroscope")
%cd /kaggle/working/neuroscope
!pip install -q nibabel pyyaml 2>/dev/null | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q tests

RAW = "/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
!ls {RAW} | head -3
!python scripts/preprocess.py --raw {RAW} --cache /kaggle/working/cache
!python scripts/train.py --cache /kaggle/working/cache --out /kaggle/working/runs/base --epochs 20
!python scripts/evaluate.py --cache /kaggle/working/cache --ckpt /kaggle/working/runs/base/best.pt --out /kaggle/working/runs/base/results

import json, glob
split = json.load(open("/kaggle/working/cache/split.json"))
demo = glob.glob(f"{RAW}/{split['test'][0]}")[0]
shutil.copytree(demo, "/kaggle/working/demo_case", dirs_exist_ok=True)
!cd /kaggle/working && zip -q -r neuroscope_artifacts.zip runs demo_case && ls -lh neuroscope_artifacts.zip
print(json.dumps(json.load(open("/kaggle/working/runs/base/results/summary.json")), indent=1)[:2500])
